In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt
import os

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
# Dataset path
DATASET_PATH = r"D:\AgriMithra datasets\PlantVillage-Dataset-master\PlantVillage-Dataset-master\raw\color"

# Maize classes only
MAIZE_CLASSES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___healthy",
    "Corn_(maize)___Northern_Leaf_Blight"
]

# Training settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 10

# Check dataset
print("Dataset exists:", os.path.exists(DATASET_PATH))

print("\nChecking maize folders:")

for class_name in MAIZE_CLASSES:
    folder_path = os.path.join(DATASET_PATH, class_name)
    print(class_name, "->", os.path.exists(folder_path))

Dataset exists: True

Checking maize folders:
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot -> True
Corn_(maize)___Common_rust_ -> True
Corn_(maize)___healthy -> True
Corn_(maize)___Northern_Leaf_Blight -> True


In [4]:
# Load training dataset - 80%
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    class_names=MAIZE_CLASSES,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load validation dataset - 20%
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    class_names=MAIZE_CLASSES,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Get class names
class_names = train_ds.class_names

print("\nClasses used for training:")

for index, class_name in enumerate(class_names):
    print(index, "->", class_name)

print("\nNumber of classes:", len(class_names))

Found 3852 files belonging to 4 classes.
Using 3082 files for training.
Found 3852 files belonging to 4 classes.
Using 770 files for validation.

Classes used for training:
0 -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
1 -> Corn_(maize)___Common_rust_
2 -> Corn_(maize)___healthy
3 -> Corn_(maize)___Northern_Leaf_Blight

Number of classes: 4


In [ ]:
#Optimize dataset loading
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("Dataset optimization completed.")

Dataset optimization completed.


In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

print("Data augmentation created.")

Data augmentation created.


In [ ]:
#Build MobileNetV2 model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the pretrained MobileNetV2 layers
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))

# Data augmentation
x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

# Pretrained MobileNetV2
x = base_model(x, training=False)

# Convert feature maps into a feature vector
x = layers.GlobalAveragePooling2D()(x)

# Reduce overfitting
x = layers.Dropout(0.2)(x)

# 4 maize classes
outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 30s 3us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ true_divide (TrueDivide)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ subtract (Subtract)                  │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4)                   │           5,124 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
#Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [9]:
MODEL_PATH = r"C:\AgriMithra\AgriMitra\backend\models\maize_model.keras"

os.makedirs(r"C:\AgriMithra\AgriMitra\backend\models", exist_ok=True)

print("Model will be saved at:", MODEL_PATH)

Model will be saved at: C:\AgriMithra\AgriMitra\backend\models\maize_model.keras


In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True
    )
]

print("Callbacks ready.")

Callbacks ready.


In [11]:
#Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/10


C:\Users\udata\anaconda3\envs\agrimitra-ml\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


97/97 ━━━━━━━━━━━━━━━━━━━━ 70s 664ms/step - accuracy: 0.8027 - loss: 0.5341 - val_accuracy: 0.9299 - val_loss: 0.2087
Epoch 2/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 59s 608ms/step - accuracy: 0.9257 - loss: 0.2059 - val_accuracy: 0.9455 - val_loss: 0.1524
Epoch 3/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 65s 670ms/step - accuracy: 0.9390 - loss: 0.1633 - val_accuracy: 0.9506 - val_loss: 0.1366
Epoch 4/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 60s 624ms/step - accuracy: 0.9338 - loss: 0.1656 - val_accuracy: 0.9390 - val_loss: 0.1459
Epoch 5/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 57s 584ms/step - accuracy: 0.9465 - loss: 0.1404 - val_accuracy: 0.9506 - val_loss: 0.1370
Epoch 6/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 63s 645ms/step - accuracy: 0.9471 - loss: 0.1446 - val_accuracy: 0.9468 - val_loss: 0.1354
Epoch 7/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 63s 654ms/step - accuracy: 0.9471 - loss: 0.1332 - val_accuracy: 0.9597 - val_loss: 0.1090
Epoch 8/10
97/97 ━━━━━━━━━━━━━━━━━━━━ 70s 717ms/step - accuracy: 0.9461 - loss: 0.1266 - val_accuracy: 0.962

In [12]:
import os

MODEL_PATH = r"D:\AgriMithra datasets\maize_model.keras"

model.save(MODEL_PATH)

print("Model saved successfully!")

Model saved successfully!


In [14]:
import tensorflow as tf
import numpy as np

IMAGE_PATH = r"D:\AgriMithra datasets\test_images\maize_test.jpg"

img = tf.keras.utils.load_img(
    "D:/CSP_images/Image-1.jpg",
    target_size=(224, 224)
)

img_array = tf.keras.utils.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array)

print(predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[[9.1205138e-01 4.6364874e-02 5.5203738e-04 4.1031726e-02]]


In [16]:
import os

DATASET_PATH = r"D:\AgriMithra datasets\PlantVillage-Dataset-master\PlantVillage-Dataset-master\raw\color"

maize_classes = sorted([
    folder for folder in os.listdir(DATASET_PATH)
    if "Corn_(maize)" in folder
])

print("Maize classes:")
for i, name in enumerate(maize_classes):
    print(i, name)

Maize classes:
0 Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
1 Corn_(maize)___Common_rust_
2 Corn_(maize)___Northern_Leaf_Blight
3 Corn_(maize)___healthy


In [1]:
import tensorflow as tf
import numpy as np

IMAGE_PATH = r"D:\CSP_images\maize2.jpg"

img = tf.keras.utils.load_img(
    "D:/CSP_images/Image-1.jpg",
    target_size=(224, 224)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array)

predicted_index = np.argmax(predictions[0])
predicted_class = maize_classes[predicted_index]
confidence = np.max(predictions[0]) * 100

print("Predicted Disease:", predicted_class)
print("Confidence:", round(confidence, 2), "%")

NameError: name 'model' is not defined